In [ ]:
print("Hello world")

In [ ]:
import asyncio
import base64
import json
import os

import sounddevice as sd
import websockets
from dotenv import load_dotenv
from elevenlabs import save
from elevenlabs.client import ElevenLabs
from IPython.display import Audio

In [ ]:
load_dotenv()
client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))
voice_id = os.getenv("ELEVENLABS_VOICE_ID")

# emotions: eleven_v3 reads inline audio tags, e.g. [excited], [whispers], [laughs], [sighs]
text = "[excited] Hi there! I'm your english teacher. [whispers] Let's practice speaking together."

save(
    client.text_to_speech.convert(
        voice_id=voice_id,
        model_id="eleven_v3",
        text=text,
    ),
    "output.mp3",
)
Audio("output.mp3")

In [ ]:
WS_URL = "wss://api.elevenlabs.io/v1/speech-to-text/realtime?model_id=scribe_v2_realtime&audio_format=pcm_16000&commit_strategy=vad"


async def main():
    async with websockets.connect(
        WS_URL, additional_headers={"xi-api-key": os.environ["ELEVENLABS_API_KEY"]}
    ) as ws:
        loop = asyncio.get_running_loop()
        queue = asyncio.Queue()

        def on_audio(indata, frames, time_info, status):
            loop.call_soon_threadsafe(queue.put_nowait, bytes(indata))

        async def send_audio():
            while True:
                chunk = await queue.get()
                await ws.send(
                    json.dumps(
                        {
                            "message_type": "input_audio_chunk",
                            "audio_base_64": base64.b64encode(chunk).decode(),
                        }
                    )
                )

        async def receive():
            async for raw in ws:
                data = json.loads(raw)
                if data.get("message_type") == "partial_transcript":
                    print(f"\r… {data['text']:<80}", end="")
                elif data.get("message_type") == "committed_transcript":
                    print(f"\r✅ {data['text']}")

        with sd.RawInputStream(
            samplerate=16000,
            channels=1,
            dtype="int16",
            blocksize=1600,
            callback=on_audio,
        ):
            print("🎤 Speak now (Ctrl+C to stop)")
            await asyncio.gather(send_audio(), receive())


await main()